# Module 4: Player Performance Analysis
**Purpose:** Calculate summary statistics for batters and bowlers from ball-by-ball delivery records.

## Shared setup — load the original project datasets
This setup is included so this notebook can run independently in a fresh Google Colab session.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

base_url = 'https://raw.githubusercontent.com/Dinesh2006-Dudi/ML-Datasets/main/archive/'
matches = pd.read_csv(base_url + 'matches.csv')
deliveries = pd.read_csv(base_url + 'deliveries.csv')
batting_stats = pd.read_csv(base_url + 'batting_stats.csv')
bowling_stats = pd.read_csv(base_url + 'bowling_stats.csv')
venues = pd.read_csv(base_url + 'venues.csv')
squads = pd.read_csv(base_url + 'squads.csv')
print('Matches:', matches.shape, '| Deliveries:', deliveries.shape)

In [ ]:
# Prepare datasets using the cleaning steps from the original notebook.
matches = matches.copy()
deliveries = deliveries.copy()

for col in ['first_ings_score', 'first_ings_wkts', 'second_ings_score', 'second_ings_wkts']:
    if col in matches.columns:
        matches[col] = matches[col].fillna(matches[col].median())
for col in ['balls_left', 'wb_runs', 'wb_wickets']:
    if col in matches.columns:
        matches[col] = matches[col].fillna(0)
for col in ['player_of_the_match', 'top_scorer', 'best_bowling']:
    if col in matches.columns:
        matches[col] = matches[col].fillna('Unknown')
if 'match_winner' in matches.columns:
    matches = matches.dropna(subset=['match_winner']).copy()

for col, value in [('wicket_type', 'No Wicket'), ('player_dismissed', 'Not Out'), ('fielder', 'No Fielder')]:
    if col in deliveries.columns:
        deliveries[col] = deliveries[col].fillna(value)
for col in ['wide', 'noballs', 'legbyes', 'byes']:
    if col in deliveries.columns:
        deliveries[col] = deliveries[col].fillna(0)

# Some later feature calculations require these numeric columns.
for col in ['runs_of_bat', 'extras']:
    if col in deliveries.columns:
        deliveries[col] = pd.to_numeric(deliveries[col], errors='coerce').fillna(0)
deliveries['total_runs'] = deliveries['runs_of_bat'] + deliveries['extras']
print('Cleaned matches:', matches.shape, '| Cleaned deliveries:', deliveries.shape)

Batsmen performance anlaysis

In [ ]:
batting_stats = deliveries.groupby('striker').agg(
    total_runs=('runs_of_bat','sum'),
    balls_faced=('runs_of_bat','count'),
    dismissals=('player_dismissed','count')
).reset_index()
batting_stats['strike_rate'] = (batting_stats['total_runs'] / batting_stats['balls_faced'] * 100).round(2)
batting_stats['average'] = (batting_stats['total_runs'] / batting_stats['dismissals'].replace(0,1)).round(2)

print("Top 5 Batsmen by Runs (Player Analytics)")
print(batting_stats.sort_values('total_runs', ascending=False).head())

Bowler performance analysis

In [ ]:
deliveries['total_runs'] = deliveries['runs_of_bat'] + deliveries['extras']
bowler_stats = deliveries.groupby('bowler').agg(
    runs_conceded=('total_runs','sum'),
    balls_bowled=('runs_of_bat','count'),
    wickets=('player_dismissed','count')
).reset_index()
bowler_stats['economy'] = (bowler_stats['runs_conceded'] / (bowler_stats['balls_bowled']/6)).round(2)
print("\nTop 5 Bowlers by Wickets")
print(bowler_stats.sort_values('wickets', ascending=False).head())